In [2]:


import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
!pip install --upgrade matplotlib seaborn
!pip uninstall -y matplotlib
!pip install matplotlib
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              confusion_matrix)

RANDOM_STATE = 42

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
})

DISTRICT_ORDER = ["Burera", "Gakenke", "Musanze", "Ngororero",
                   "Nyabihu", "Rubavu", "Rutsiro"]
SEASON_ORDER = ["Season A (Rainy)", "Season B (Rainy)", "Dry Season"]
CATEGORY_ORDER = ["Below Normal Rainfall", "Normal Rainfall", "Above Normal Rainfall"]
CATEGORY_COLORS = {"Below Normal Rainfall": "#D9A441",
                    "Normal Rainfall": "#4CAF91",
                    "Above Normal Rainfall": "#2E86AB"}

df = pd.read_csv("Processed Data.csv", parse_dates=["date"])
df["grid_cell_id"] = df["Latitude"].astype(str) + "_" + df["Longitude"].astype(str)

# ==========================================================================
# STEP 1: AGGREGATE TO DISTRICT-YEAR-SEASON LEVEL
# ==========================================================================
print("=" * 150)
print("STEP 1: DISTRICT-YEAR-SEASON LEVEL AGGREGATION")
print("=" * 150)

grid_seasonal = (
    df.groupby(["District", "grid_cell_id", "Year", "Season"])
    .agg(pr_seasonal_total=("pr", "sum"),
         tas_seasonal_mean=("tas", "mean"),
         tasmin_seasonal_mean=("tasmin", "mean"),
         tasmax_seasonal_mean=("tasmax", "mean"),
         temp_range_seasonal_mean=("temp_range", "mean"))
    .reset_index()
)

seasonal = (
    grid_seasonal.groupby(["District", "Year", "Season"])
    .agg(pr_seasonal_total=("pr_seasonal_total", "mean"),
         tas_seasonal_mean=("tas_seasonal_mean", "mean"),
         tasmin_seasonal_mean=("tasmin_seasonal_mean", "mean"),
         tasmax_seasonal_mean=("tasmax_seasonal_mean", "mean"),
         temp_range_seasonal_mean=("temp_range_seasonal_mean", "mean"))
    .reset_index()
)
seasonal = seasonal.sort_values(["District", "Season", "Year"]).reset_index(drop=True)

print(f"Aggregated dataset shape: {seasonal.shape} "
      f"(7 districts x 39 years x 3 seasons = {7*39*3} expected)")

agg_summary = (
    seasonal.groupby(["District", "Season"])["pr_seasonal_total"]
    .agg(N="count", Mean="mean", Std="std", Min="min", Max="max")
    .round(1).reset_index()
)
print(agg_summary.to_string(index=False))
agg_summary.to_csv("Seasonal Aggregated Summary.csv", index=False)

# ==========================================================================
# STEP 2: TERCILE METHOD APPLICATION FOR SEASONAL RAINFALL CATEGORY CLASSIFICATION
# ==========================================================================
print("\n" + "=" * 150)
print("STEP 2: TERCILE CLASSIFICATION")
print("=" * 150)

threshold_rows = []
seasonal["Rainfall_Category"] = None
for (d, s_), group in seasonal.groupby(["District", "Season"]):
    p33 = group["pr_seasonal_total"].quantile(1 / 3)
    p67 = group["pr_seasonal_total"].quantile(2 / 3)
    idx = group.index
    cat = pd.cut(
        seasonal.loc[idx, "pr_seasonal_total"],
        bins=[-np.inf, p33, p67, np.inf],
        labels=CATEGORY_ORDER
    )
    seasonal.loc[idx, "Rainfall_Category"] = cat
    threshold_rows.append({
        "District": d, "Season": s_,
        "33rd Percentile (mm)": round(p33, 1),
        "67th Percentile (mm)": round(p67, 1),
    })

threshold_table = pd.DataFrame(threshold_rows)
print(threshold_table.to_string(index=False))
threshold_table.to_csv("District Tercile Thresholds.csv", index=False)

category_dist = seasonal.groupby(["Season", "Rainfall_Category"], observed=True).size().reset_index(name="Count")
category_dist_pivot = category_dist.pivot(index="Season", columns="Rainfall_Category", values="Count").reindex(SEASON_ORDER)[CATEGORY_ORDER]
print("\nRainfall Category Distribution:")
print(category_dist_pivot.to_string())
category_dist_pivot.to_csv("Rainfall Category Distribution.csv")

# ==========================================================================
# STEP 3: FEATURE ENGINEERING - LAGGED FEATURES
# ==========================================================================
print("\n" + "=" * 140)
print("STEP 3: Lagged Features Generation")
print("=" * 140)

lag_source_cols = ["pr_seasonal_total", "tas_seasonal_mean",
                    "tasmin_seasonal_mean", "tasmax_seasonal_mean",
                    "temp_range_seasonal_mean"]
lag_new_names = ["Prior Year Rainfall (mm) ", "Prior Year Mean Tas ",
                  "Prior Year Mean Tasmin ", "Prior Year Mean Tasmax",
                  "Prior Year Mean Temp Range"]

seasonal = seasonal.sort_values(["District", "Season", "Year"])
for src, new in zip(lag_source_cols, lag_new_names):
    seasonal[new] = seasonal.groupby(["District", "Season"])[src].shift(1)

n_before = len(seasonal)
model_df = seasonal.dropna(subset=lag_new_names).copy()
n_after = len(model_df)
print(f"Rows without prior year data: {n_before}")
print(f"Rows after first year dropping/ District X Season Group: {n_after}")
print(f"Dropped {n_before - n_after} rows (the first year, 1983, "
      f"21 district-season groups without prior "
      f"year to lag from cannot be used for predictions).")

district_dummies = pd.get_dummies(model_df["District"], prefix="District", dtype=int)
season_dummies = pd.get_dummies(model_df["Season"], prefix="Season", dtype=int)
model_df = pd.concat([model_df, district_dummies, season_dummies], axis=1)

FEATURE_COLS = (
    lag_new_names + ["Year"]
    + list(district_dummies.columns) + list(season_dummies.columns)
)
print(f"\nFinal feature set ({len(FEATURE_COLS)}): {FEATURE_COLS}")

X = model_df[FEATURE_COLS].copy()
y = model_df["Rainfall_Category"].astype(str).copy()

# ==========================================================================
# STEP 4: CHRONOLOGICAL TRAIN / TEST-SPLIT
# ==========================================================================
print("\n" + "=" * 150)
print("STEP 4: CHRONOLOGICAL TRAIN / TEST-SPLIT")
print("=" * 150)

# ============================================================
# CHRONOLOGICAL TRAIN-TEST SPLIT
# ============================================================

# 1. Sort observations chronologically

date_order = pd.to_datetime(df.loc[X.index, "date"])

sorted_indices = date_order.sort_values(kind="stable").index

X = X.loc[sorted_indices]
y = y.loc[sorted_indices]

# 2. Chronological split
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# 3. Distribution summary - for Seasonal Rainfall category
split_rows = []

for label, series in [("Train", y_train), ("Test", y_test)]:
    for c in CATEGORY_ORDER:
        n = (series == c).sum()
        split_rows.append({
            "Split": label,
            "Category": c,
            "N": int(n),
            "Percent of Split (%)": round(n / len(series) * 100, 1)
        })

split_summary = pd.DataFrame(split_rows)

print("\n TRAIN-TEST SPLIT SUMMARY")
print("=" * 70)
print(split_summary.to_string(index=False))

# 4. Split summary
split_summary.to_csv("Train-Test-Split Summary.csv", index=False)

print("\nCHRONOLOGICAL SPLIT")
print("=" * 70)

print(f"Total Observations: {len(X)}")

print(f"Total Training Observations: {len(X_train)} "
      f"({len(X_train)/len(X)*100:.1f}%)")

print(f"Total Test Observations    : {len(X_test)} "
      f"({len(X_test)/len(X)*100:.1f}%)")

print("\nTraining Period:",
      date_order.loc[X_train.index].min(),
      "to",
      date_order.loc[X_train.index].max())

print("Testing Period :",
      date_order.loc[X_test.index].min(),
      "to",
      date_order.loc[X_test.index].max())

print("\nChronological split completed Successfuly.")

# ==========================================================================
# STEP 5: MODEL TRAINING
# ==========================================================================
print("\n" + "=" * 140)
print("STEP 5: Model Training")
print("=" * 140)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "SVM (RBF Kernel)": SVC(
        kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE
    ),
}

predictions = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions[name] = model.predict(X_test_scaled)
    print(f"-> {name} trained.")

# ==========================================================================
# STEP 6: MODEL PERFORMANCE COMPARISON
# ==========================================================================
print("\n" + "=" * 140)
print("MODEL PERFORMANCE")
print("=" * 140)

perf_rows = []
for name, y_pred in predictions.items():
    perf_rows.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision (macro)": round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Recall (macro)": round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "F1-Score (macro)": round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "F1-Score (weighted)": round(f1_score(y_test, y_pred, average="weighted", zero_division=0), 4),
    })
performance_summary = pd.DataFrame(perf_rows).sort_values("F1-Score (macro)", ascending=False)
print(performance_summary.to_string(index=False))
performance_summary.to_csv("Model Performance.csv", index=False)

naive_baseline_acc = max((y_test == c).mean() for c in CATEGORY_ORDER)
print(f"\nNaive baseline accuracy: "
      f"{naive_baseline_acc:.4f} -- For Model Comparison.")

# ==========================================================================
# STEP 7: PER-CLASS CLASSIFICATION REPORT
# ==========================================================================
report_rows = []
for name, y_pred in predictions.items():
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    for c in CATEGORY_ORDER:
        if c in report:
            r = report[c]
            report_rows.append({
                "Model": name, "Category": c,
                "Precision": round(r["precision"], 3),
                "Recall": round(r["recall"], 3),
                "F1-Score": round(r["f1-score"], 3),
                "Support": int(r["support"]),
            })
classification_long = pd.DataFrame(report_rows)
print("\n" + classification_long.to_string(index=False))
classification_long.to_csv("Extended Classification Report.csv", index=False)

# ==========================================================================
# STEP 8: RANDOM FOREST FEATURE IMPORTANCE
# ==========================================================================
rf_model = models["Random Forest"]
importance_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE")
print("=" * 140)
print(importance_df.to_string(index=False))
importance_df.to_csv("Feature Importance- RF.csv", index=False)

# ==========================================================================
# FIGURES
# ==========================================================================

fig, axes = plt.subplots(3, 1, figsize=(11, 13), sharex=True)
for ax, s_ in zip(axes, SEASON_ORDER):
    sub = seasonal[seasonal["Season"] == s_]
    for d in DISTRICT_ORDER:
        d_sub = sub[sub["District"] == d].sort_values("Year")
        ax.plot(d_sub["Year"], d_sub["pr_seasonal_total"], marker="o",
                markersize=3, linewidth=1, alpha=0.8, label=d)
    ax.set_title(f"{s_}")
    ax.set_ylabel("Seasonal Rainfall Total (mm)")
    if s_ == SEASON_ORDER[0]:
        ax.legend(loc="upper left", fontsize=8, ncol=4)
axes[-1].set_xlabel("Year")
fig.suptitle("District Seasonal Rainfall Totals (1983-2021)",
             fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig("District Seasonal Total Trends.png")
plt.close()
print("\nSaved: District Seasonal Total Trends.png")

fig, ax = plt.subplots(figsize=(9, 5.5))
category_dist_pivot.plot(kind="bar", stacked=False, ax=ax,
                          color=[CATEGORY_COLORS[c] for c in CATEGORY_ORDER])
ax.set_title("Category Distribution by Season (1983-2021))")
ax.set_xlabel("Season")
ax.set_ylabel("District-Years)")
ax.legend(title="Category")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("Category Distribution.png")
plt.close()
print("Saved: Category Distribution.png")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
short_labels = ["Below", "Normal", "Above"]
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=CATEGORY_ORDER)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=short_labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
    ax.set_xlabel("Forecasted Category")
    ax.set_ylabel("Actual Category")

fig.suptitle("Confusion Matrices: Seasonal Rainfall Category Forecast",
             fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig("Confusion Matrices.png")
plt.close()
print("Saved: Confusion Matrices.png")

fig, ax = plt.subplots(figsize=(9, 5.5))
metrics_to_plot = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-Score (macro)"]
plot_df = performance_summary.set_index("Model")[metrics_to_plot]
plot_df.plot(kind="bar", ax=ax, color=["#2E86AB", "#4CAF91", "#D9A441", "#D64545"])
ax.axhline(naive_baseline_acc, color="black", linestyle=":", linewidth=1.5,
           label=f"Naive baseline accuracy ({naive_baseline_acc:.2f})")
ax.set_title("Model Performance Comparison")

ax.set_ylabel("Score Value")
ax.set_ylim(0, 1)
ax.legend(loc="lower right", fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("Model Performance.png")
plt.close()
print("Saved: Model Performance.png")

fig, ax = plt.subplots(figsize=(9, 6.5))
sns.barplot(data=importance_df, x="Importance", y="Feature", color="#2E86AB", ax=ax)
ax.set_title("Random Forest Feature Importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("Feature Importance.png")
plt.close()
print("Saved: Feature Importance.png")

print("\n" + "=" * 150)
print("SEASONAL RAINFALL MODELING SUCCESS")
print(f"Best model by macro F1-score Performance: {performance_summary.iloc[0]['Model']} "
      f"({performance_summary.iloc[0]['F1-Score (macro)']})")
print(f"Naive baseline accuracy for comparison: {naive_baseline_acc:.4f}")
print("=" * 150)


Found existing installation: matplotlib 3.11.2
Uninstalling matplotlib-3.11.2:
  Successfully uninstalled matplotlib-3.11.2
  Using cached matplotlib-3.11.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (80 kB)
Using cached matplotlib-3.11.2-cp311-cp311-macosx_11_0_arm64.whl (9.3 MB)
STEP 1: DISTRICT-YEAR-SEASON LEVEL AGGREGATION
Aggregated dataset shape: (819, 8) (7 districts x 39 years x 3 seasons = 819 expected)
 District           Season  N  Mean   Std   Min   Max
   Burera       Dry Season 39 273.6  79.5  77.8 412.3
   Burera Season A (Rainy) 39 525.4  87.1 248.2 678.5
   Burera Season B (Rainy) 39 485.6 109.6 125.0 672.7
  Gakenke       Dry Season 39 347.4 130.3  93.8 686.2
  Gakenke Season A (Rainy) 39 547.4 101.8 315.0 712.5
  Gakenke Season B (Rainy) 39 541.0 113.6 293.0 726.7
  Musanze       Dry Season 39 305.7 109.8  78.8 621.0
  Musanze Season A (Rainy) 39 536.2  89.6 316.2 658.8
  Musanze Season B (Rainy) 39 481.3 108.9 199.4 709.4
Ngororero       Dry Season 39 323.1 104.3 13